# Environment (Run First Section)

In [ ]:
!pip install transformers datasets py7zr evaluate rouge_score tqdm --quiet

In [ ]:
!pip install --upgrade --quiet langchain-text-splitters tiktoken

In [ ]:
from langchain_text_splitters import TokenTextSplitter
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain.text_splitter import TokenTextSplitter
from google.colab import files
import json
from huggingface_hub import notebook_login
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
import evaluate
from rouge_score import rouge_scorer
import nltk
from langchain.chains.summarize import load_summarize_chain
from langchain_huggingface import HuggingFacePipeline
from tqdm.notebook import tqdm

In [ ]:
notebook_login()

In [ ]:
nltk.download('punkt')

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

In [ ]:
datasets = [load_dataset("ccdv/arxiv-summarization"), load_dataset("ccdv/pubmed-summarization"), load_dataset("ccdv/govreport-summarization")]
datasets

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[DatasetDict({
     train: Dataset({
         features: ['article', 'abstract'],
         num_rows: 203037
     })
     validation: Dataset({
         features: ['article', 'abstract'],
         num_rows: 6436
     })
     test: Dataset({
         features: ['article', 'abstract'],
         num_rows: 6440
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['article', 'abstract'],
         num_rows: 119924
     })
     validation: Dataset({
         features: ['article', 'abstract'],
         num_rows: 6633
     })
     test: Dataset({
         features: ['article', 'abstract'],
         num_rows: 6658
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['report', 'summary'],
         num_rows: 17517
     })
     validation: Dataset({
         features: ['report', 'summary'],
         num_rows: 973
     })
     test: Dataset({
         features: ['report', 'summary'],
         num_rows: 973
     })
 })]

# Testing (Don't need to run)

In [ ]:
for i, ds in enumerate(datasets):
  if i == 2:
    test_summaries = [ds['test'][i]['summary'] for i in range(len(ds['test']))]
  else:
    test_summaries = [ds['test'][i]['abstract'] for i in range(len(ds['test']))]
  lens = [len(sum) for sum in test_summaries]
  print(f"Dataset {i}: {sum(lens)/len(lens)}")

Dataset 0: 966.4517080745342
Dataset 1: 1252.1004806248122
Dataset 2: 3851.4850976361768


In [ ]:
facebook_summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
fb_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
google_summarizer = pipeline("summarization", model="google/pegasus-large")
google_tokenizer = AutoTokenizer.from_pretrained("google/pegasus-large")
t5_summarizer = pipeline("summarization", model="Falconsai/text_summarization")
t5_tokenizer = AutoTokenizer.from_pretrained("Falconsai/text_summarization")

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def first_tokens(article):
    bart_tokens_tensor = fb_tokenizer(article, return_tensors="pt").input_ids[0][:512]
    pegasus_tokens_tensor = google_tokenizer(article, return_tensors="pt").input_ids[0][:512]
    t5_tokens_tensor = t5_tokenizer(article, return_tensors="pt").input_ids[0][:512]
    bart_tokens = bart_tokens_tensor.tolist()
    pegasus_tokens = pegasus_tokens_tensor.tolist()
    t5_tokens = t5_tokens_tensor.tolist()
    bart_tokens = fb_tokenizer.convert_tokens_to_string(fb_tokenizer.convert_ids_to_tokens(bart_tokens))
    pegasus_tokens = google_tokenizer.convert_tokens_to_string(google_tokenizer.convert_ids_to_tokens(pegasus_tokens))
    t5_tokens = t5_tokenizer.convert_tokens_to_string(t5_tokenizer.convert_ids_to_tokens(t5_tokens))
    return bart_tokens, pegasus_tokens, t5_tokens


In [ ]:
results = {"datasets": [{"name": "ccdv/arxiv-summarization", "models": {"bert": {}, "pegasus": {}, "t5": {}}}, {"name": "ccdv/pubmed-summarization", "models": {"bert": {}, "pegasus": {}, "t5": {}}}, {"name": "ccdv/govreport-summarization", "models": {"bert": {}, "pegasus": {}, "t5": {}}}]}

In [ ]:
test_indexes = [0, 10, 20]
rouge_scores = {"fb":[], "google":[], "t5":[]}
for i, ds in enumerate(datasets):
  print(f"{i}  ", end='\r')
  if i == 2:
    article_text = "report"
    sum_text = "summary"
  else:
    article_text = "article"
    sum_text = "abstract"
  for ex_idx in test_indexes:
    print(i, ex_idx)
    article = ds['test'][ex_idx][article_text]
    true_summary = ds['test'][ex_idx][sum_text]
    bart_article, pegasus_article, t5_article = first_tokens(article)
    summaries = {"fb":[], "google":[], "t5":[]}
    fb_summary = str(facebook_summarizer(bart_article, max_length=300, min_length=25, do_sample=False)[0]["summary_text"])
    google_summary = str(google_summarizer(pegasus_article, max_length=300, min_length=25, do_sample=False)[0]["summary_text"])
    t5_summary = str(t5_summarizer(t5_article, max_length=300, min_length=25, do_sample=False)[0]["summary_text"])
    results["datasets"][i]["models"]["bert"][ex_idx] = scorer.score(true_summary, fb_summary)
    results["datasets"][i]["models"]["pegasus"][ex_idx] = scorer.score(true_summary, google_summary)
    results["datasets"][i]["models"]["t5"][ex_idx] = scorer.score(true_summary, t5_summary)
    print("t5", t5_summary)
    print("true", true_summary)
results


0 0
t5 for about 20 years the problem of properties of short- term changes of solar activity has been considered extensively . many investigators studied the short - term periodicities of the various indices of solar activities . several periodicities were detected , but the periodicities about 155 days and from the interval of @xmath3 $ $ ] days ( @Xmath4 $ $] years ) are mentioned most often . first of them was discovered by @xcite in the occurence rate of gamma -
true the short - term periodicities of the daily sunspot area fluctuations from august 1923 to october 1933 are discussed . for these data 
 the correlative analysis indicates negative correlation for the periodicity of about @xmath0 days , but the power spectrum analysis indicates a statistically significant peak in this time interval . 
 a new method of the diagnosis of an echo - effect in spectrum is proposed and it is stated that the 155-day periodicity is a harmonic of the periodicities from the interval of @xmath1 $ ]

{'datasets': [{'name': 'ccdv/arxiv-summarization',
   'models': {'bert': {0: {'rouge1': Score(precision=0.6829268292682927, recall=0.1407035175879397, fmeasure=0.23333333333333334),
      'rouge2': Score(precision=0.25, recall=0.050505050505050504, fmeasure=0.08403361344537816),
      'rougeL': Score(precision=0.5121951219512195, recall=0.10552763819095477, fmeasure=0.175)},
     10: {'rouge1': Score(precision=0.4, recall=0.16, fmeasure=0.22857142857142856),
      'rouge2': Score(precision=0.06896551724137931, recall=0.02702702702702703, fmeasure=0.03883495145631068),
      'rougeL': Score(precision=0.23333333333333334, recall=0.09333333333333334, fmeasure=0.13333333333333333)},
     20: {'rouge1': Score(precision=0.4791666666666667, recall=0.15862068965517243, fmeasure=0.2383419689119171),
      'rouge2': Score(precision=0.0851063829787234, recall=0.027777777777777776, fmeasure=0.041884816753926704),
      'rougeL': Score(precision=0.2916666666666667, recall=0.09655172413793103, fmeas

In [ ]:
rows = []
for dataset in results['datasets']:
    dataset_name = dataset['name']
    for model_name, indices in dataset['models'].items():
        for index, scores in indices.items():
            rouge1_scores = scores['rouge1']
            rouge2_scores = scores['rouge2']
            rougeL_scores = scores['rougeL']
            rows.append({
                'Dataset': dataset_name,
                'Model': model_name,
                'Index': index,
                'Rouge1_Precision': rouge1_scores[0],
                'Rouge1_Recall': rouge1_scores[1],
                'Rouge1_FMeasure': rouge1_scores[2],
                'Rouge2_Precision': rouge2_scores[0],
                'Rouge2_Recall': rouge2_scores[1],
                'Rouge2_FMeasure': rouge2_scores[2],
                'RougeL_Precision': rougeL_scores[0],
                'RougeL_Recall': rougeL_scores[1],
                'RougeL_FMeasure': rougeL_scores[2],
            })
df1 = pd.DataFrame(rows)

In [ ]:
gdfs1 = df1.groupby(by=["Dataset", "Model"])
desc_stats1 = {}
for i, gdf in gdfs1:
    desc_stats1[i] = gdf.describe()
keys = list(desc_stats1.keys())
keys

[('ccdv/arxiv-summarization', 'bert'),
 ('ccdv/arxiv-summarization', 'pegasus'),
 ('ccdv/arxiv-summarization', 't5'),
 ('ccdv/govreport-summarization', 'bert'),
 ('ccdv/govreport-summarization', 'pegasus'),
 ('ccdv/govreport-summarization', 't5'),
 ('ccdv/pubmed-summarization', 'bert'),
 ('ccdv/pubmed-summarization', 'pegasus'),
 ('ccdv/pubmed-summarization', 't5')]

In [ ]:
desc_stats1[('ccdv/pubmed-summarization', 't5')]

,Index,Rouge1_Precision,Rouge1_Recall,Rouge1_FMeasure,Rouge2_Precision,Rouge2_Recall,Rouge2_FMeasure,RougeL_Precision,RougeL_Recall,RougeL_FMeasure
count,3.0,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,10.0,0.479825,0.102539,0.157933,0.189834,0.031548,0.049908,0.274561,0.061234,0.093705
std,10.0,0.149707,0.067535,0.082615,0.155880,0.017236,0.021561,0.070433,0.044640,0.055986
min,0.0,0.368421,0.050193,0.093190,0.081081,0.016949,0.028037,0.210526,0.027027,0.050179
25%,5.0,0.394737,0.064422,0.111410,0.100541,0.022040,0.039289,0.236842,0.035985,0.062127
50%,10.0,0.421053,0.078652,0.129630,0.120000,0.027132,0.050542,0.263158,0.044944,0.074074
75%,15.0,0.535526,0.128711,0.190305,0.244211,0.038847,0.060844,0.306579,0.078338,0.115468
max,20.0,0.650000,0.178771,0.250980,0.368421,0.050562,0.071146,0.350000,0.111732,0.156863


# LangChain Testing (Don't need to run)
___________________________________________

In [ ]:
!pip install langchain huggingface_hub --quiet

In [ ]:
pipe = pipeline("summarization", model="facebook/bart-large-cnn")
hf = HuggingFacePipeline(pipeline=pipe)
hf

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


HuggingFacePipeline(pipeline=<transformers.pipelines.text2text_generation.SummarizationPipeline object at 0x792222bfc250>, model_id='facebook/bart-large-cnn')

In [ ]:
test_article = str(datasets[0]['test'][0]['article']) + "\nSummary:\n"

In [ ]:
bart_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

In [ ]:
tokens = tokenizer.tokenize(test_article)
print(f"Total Tokens: {len(tokens)}")

Total Tokens: 8052


In [ ]:
!pip install --upgrade --quiet langchain-text-splitters tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.2 MB/s eta 0:00:00


In [ ]:
len(tokenizer.tokenize(f"Summarize the following chunk of an article:\n\nSummary:\n"))

15

In [ ]:
text_splitter = TokenTextSplitter(chunk_size=497, chunk_overlap=64)

texts = text_splitter.split_text(test_article)
for text in texts:
  print(len(tokenizer.tokenize(text)))

497
497
497
497
497
497
497
497
497
497
497
497
497
497
497
497
497
497
258


In [ ]:
summary = []
for text in texts:
  prompt = f"Summarize the following chunk of an article:\n{text}\nSummary:\n"
  input_ids = tokenizer(prompt, return_tensors="pt").input_ids
  output = bart_model.generate(input_ids, max_new_tokens = int(512/len(texts)))[0]
  summary.append(tokenizer.decode(output, skip_special_tokens=True))
summary

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1399: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (27). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 27, its default value.
  warnings.warn(


['For about 20 years the problem of properties of short - term changes of solar activity has been considered extensively. The periodicities',
 'During 1964 - 2000 the sunspot number wavelet power of periods less than one year shows a cyclic evolution with the',
 'The authors concluded that the length of this period is variable and the reason of the periodicity is still not understood.',
 'The bt method can be used to verify a reality of peaks which are computed using a method giving the better resolution.',
 'The method of the diagnosis of an echo - effect in the power spectrum ( de) consists in an analysis of a period',
 'The de method analyses raw estimators of the power spectrum. The fisher test checks the null hypothesis that the time series is',
 'Summarize the following chunk of an article: The index @xmath44 indicates successive parts of the cosinus function',
 'Summarize the following chunk of an article. The index @xmath65 describes a percentage of the contribution of the',
 '

In [ ]:
summary = " ".join(summary)
input_ids = tokenizer(summary, return_tensors="pt").input_ids
output = bart_model.generate(input_ids, max_new_tokens = 512)[0]
gen_summ = tokenizer.decode(output, skip_special_tokens=True)
scorer.score(datasets[0]['test'][0]['abstract'], gen_summ)

{'rouge1': Score(precision=0.5319148936170213, recall=0.12562814070351758, fmeasure=0.20325203252032517),
 'rouge2': Score(precision=0.1956521739130435, recall=0.045454545454545456, fmeasure=0.0737704918032787),
 'rougeL': Score(precision=0.3617021276595745, recall=0.08542713567839195, fmeasure=0.13821138211382114)}

# Final Testing (Run)
____________________________________________

In [ ]:
bert_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")
bert_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
pegasus_model = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-large")
pegasus_tokenizer = AutoTokenizer.from_pretrained("google/pegasus-large")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("Falconsai/text_summarization")
t5_tokenizer = AutoTokenizer.from_pretrained("Falconsai/text_summarization")

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
device = "cuda"
bert_model.to(device)
pegasus_model.to(device)
t5_model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
test_indexes = list(range(0, 500, 10))
small_text_splitter = TokenTextSplitter(chunk_size=497, chunk_overlap=64)
large_text_splitter = TokenTextSplitter(chunk_size=990, chunk_overlap=128)


In [ ]:
stored_data = {"datasets": [{"name": "ccdv/arxiv-summarization", "models": {"bert": {}, "pegasus": {}, "t5": {}}}, {"name": "ccdv/pubmed-summarization", "models": {"bert": {}, "pegasus": {}, "t5": {}}}, {"name": "ccdv/govreport-summarization", "models": {"bert": {}, "pegasus": {}, "t5": {}}}]}


In [ ]:

for i, dataset in enumerate(datasets):
  if i == 2:
    article_text = "report"
    sum_text = "summary"
  else:
    article_text = "article"
    sum_text = "abstract"
  for ex_idx in tqdm(test_indexes, desc=f"Dataset{i+1}"):
    article = dataset['test'][ex_idx][article_text]
    true_summary = dataset['test'][ex_idx][sum_text]
    chunks = small_text_splitter.split_text(article)
    summaries = {"bert":[], "pegasus":[], "t5":[]}
    for j, chunk in enumerate(chunks):
      prompt = f"Summarize the following chunk of an article:\n{chunk}\nSummary:\n"
      bert_input_ids = bert_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
      bert_output = bert_model.generate(bert_input_ids, max_new_tokens = int(512/len(chunks)))[0]
      summaries['bert'].append(bert_tokenizer.decode(bert_output, skip_special_tokens=True))
      pegasus_input_ids = pegasus_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
      pegasus_output = pegasus_model.generate(pegasus_input_ids, max_new_tokens = int(512/len(chunks)))[0]
      summaries['pegasus'].append(pegasus_tokenizer.decode(pegasus_output, skip_special_tokens=True))
      t5_input_ids = t5_tokenizer(prompt, return_tensors="pt").input_ids.to(device)
      t5_output = t5_model.generate(t5_input_ids, max_new_tokens = int(512/len(chunks)))[0]
      summaries['t5'].append(t5_tokenizer.decode(t5_output, skip_special_tokens=True))
    bert_summary = " ".join(summaries['bert'])
    bert_input_ids = bert_tokenizer(bert_summary, return_tensors="pt").input_ids.to(device)
    bert_output = bert_model.generate(bert_input_ids, max_new_tokens = 512)[0]
    bert_gen_summ = bert_tokenizer.decode(bert_output, skip_special_tokens=True)
    pegasus_summary = " ".join(summaries['pegasus'])
    pegasus_input_ids = pegasus_tokenizer(pegasus_summary, return_tensors="pt").input_ids.to(device)
    pegasus_output = pegasus_model.generate(pegasus_input_ids, max_new_tokens = 512)[0]
    pegasus_gen_summ = pegasus_tokenizer.decode(pegasus_output, skip_special_tokens=True)
    t5_summary = " ".join(summaries['t5'])
    t5_input_ids = t5_tokenizer(t5_summary, return_tensors="pt").input_ids.to(device)
    t5_output = t5_model.generate(t5_input_ids, max_new_tokens = 512)[0]
    t5_gen_summ = t5_tokenizer.decode(t5_output, skip_special_tokens=True)
    stored_data["datasets"][i]["models"]["bert"][ex_idx] = scorer.score(true_summary, bert_gen_summ)
    stored_data["datasets"][i]["models"]["pegasus"][ex_idx] = scorer.score(true_summary, pegasus_gen_summ)
    stored_data["datasets"][i]["models"]["t5"][ex_idx] = scorer.score(true_summary, t5_gen_summ)
with open('model_data.json', 'w') as f:
  json.dump(stored_data, f)
files.download('model_data.json')

Dataset1:   0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1399: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (43). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 43, its default value.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1399: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (8). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 8, its default value.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1399: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (37). Generation will stop at the defi

Dataset2:   0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1399: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (15). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 15, its default value.
  warnings.warn(


Dataset3:   0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1399: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (23). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 23, its default value.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1399: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (14). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 14, its default value.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1399: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (9). Generation will stop at the def

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
rows = []
for dataset in stored_data['datasets']:
    dataset_name = dataset['name']
    for model_name, indices in dataset['models'].items():
        for index, scores in indices.items():
            rouge1_scores = scores['rouge1']
            rouge2_scores = scores['rouge2']
            rougeL_scores = scores['rougeL']
            rows.append({
                'Dataset': dataset_name,
                'Model': model_name,
                'Index': index,
                'Rouge1_Precision': rouge1_scores[0],
                'Rouge1_Recall': rouge1_scores[1],
                'Rouge1_FMeasure': rouge1_scores[2],
                'Rouge2_Precision': rouge2_scores[0],
                'Rouge2_Recall': rouge2_scores[1],
                'Rouge2_FMeasure': rouge2_scores[2],
                'RougeL_Precision': rougeL_scores[0],
                'RougeL_Recall': rougeL_scores[1],
                'RougeL_FMeasure': rougeL_scores[2],
            })
df = pd.DataFrame(rows)
df.to_csv('model_data.csv', index=False)
files.download('model_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
gdfs = df.groupby(by=["Dataset", "Model"])

In [ ]:
desc_stats = {}
for i, gdf in gdfs:
    desc_stats[i] = gdf.describe()

In [ ]:
keys = list(desc_stats.keys())
keys

[('ccdv/arxiv-summarization', 'bert'),
 ('ccdv/arxiv-summarization', 'pegasus'),
 ('ccdv/arxiv-summarization', 't5'),
 ('ccdv/govreport-summarization', 'bert'),
 ('ccdv/govreport-summarization', 'pegasus'),
 ('ccdv/govreport-summarization', 't5'),
 ('ccdv/pubmed-summarization', 'bert'),
 ('ccdv/pubmed-summarization', 'pegasus'),
 ('ccdv/pubmed-summarization', 't5')]

In [ ]:
desc_stats[('ccdv/pubmed-summarization', 't5')]

,Index,Rouge1_Precision,Rouge1_Recall,Rouge1_FMeasure,Rouge2_Precision,Rouge2_Recall,Rouge2_FMeasure,RougeL_Precision,RougeL_Recall,RougeL_FMeasure
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,245.000000,0.443864,0.170153,0.226915,0.126483,0.054377,0.068317,0.286691,0.114209,0.149441
std,145.773797,0.142650,0.112642,0.090109,0.098388,0.095064,0.080368,0.101201,0.104542,0.082218
min,0.000000,0.100000,0.047244,0.064171,0.000000,0.000000,0.000000,0.083333,0.039370,0.053476
25%,122.500000,0.350870,0.111342,0.169019,0.056818,0.019648,0.027663,0.205834,0.072067,0.105131
50%,245.000000,0.440339,0.137362,0.205689,0.097407,0.029680,0.042283,0.288312,0.082063,0.123813
75%,367.500000,0.574364,0.189211,0.273917,0.186921,0.061157,0.077794,0.358154,0.118856,0.159716
max,490.000000,0.640000,0.765957,0.585366,0.400000,0.652174,0.495868,0.520000,0.744681,0.569106


In [ ]:
for key in keys:
  table = desc_stats[key]
  r1 = table.loc['mean', 'Rouge1_FMeasure']
  r2 = table.loc['mean', 'Rouge2_FMeasure']
  rl = table.loc['mean', 'RougeL_FMeasure']
  print(f"{key}: {round(r1, 2)} / {round(r2, 2)} / {round(rl, 2)}")

('ccdv/arxiv-summarization', 'bert'): 0.26 / 0.07 / 0.16
('ccdv/arxiv-summarization', 'pegasus'): 0.33 / 0.09 / 0.18
('ccdv/arxiv-summarization', 't5'): 0.17 / 0.03 / 0.12
('ccdv/govreport-summarization', 'bert'): 0.12 / 0.05 / 0.09
('ccdv/govreport-summarization', 'pegasus'): 0.34 / 0.12 / 0.16
('ccdv/govreport-summarization', 't5'): 0.15 / 0.05 / 0.1
('ccdv/pubmed-summarization', 'bert'): 0.24 / 0.07 / 0.15
('ccdv/pubmed-summarization', 'pegasus'): 0.28 / 0.09 / 0.16
('ccdv/pubmed-summarization', 't5'): 0.23 / 0.07 / 0.15


# Other
____________________________________________________

Calculating approximate average token length of articles

In [ ]:
for j in range(3):
  l = []
  for i in range(0,6400):
    if j == 2:
      article = datasets[j]['test'][i]['report']
    else:
      article = datasets[j]['test'][i]['article']
    num_tokens = len(bert_tokenizer(article).input_ids)
    l.append(num_tokens)
    if j == 2 and i == 900:
      break
  print(sum(l)/len(l))


8540.76265625
4080.09125
9204.471698113208
